# **SETUP CANONICAL CV SCHEME**

In [20]:
import pathlib
import pandas as pd
from sklearn.model_selection import KFold

SEED = 42
N_SPLITS = 5
PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

nested_cv_df = pd.DataFrame({"id": train.index})
nested_cv_df["outer_fold"] = -1
for col in [f"inner_fold_{k}" for k in range(N_SPLITS)]:
    nested_cv_df[col] = -1

for outer_k, (outer_train_idx, outer_test_idx) in enumerate(kf.split(train)):
    nested_cv_df.iloc[outer_test_idx, nested_cv_df.columns.get_loc("outer_fold")] = outer_k
    col = f"inner_fold_{outer_k}"
    for inner_k, (_, inner_val_idx) in enumerate(kf.split(train.iloc[outer_train_idx])):
        original_idx = outer_train_idx[inner_val_idx]
        nested_cv_df.iloc[original_idx, nested_cv_df.columns.get_loc(col)] = inner_k

nested_cv_df = nested_cv_df.set_index("id").astype("int8")
nested_cv_df.to_parquet(PROJECT_ROOT / "data" / "cv.parquet")